# 内容简介

一个肺癌案例，串起假设检验与常见机器学习模型。

首先我会对代谢组学进行基本介绍，侧重关注的科学问题、分析方法、数据结构；然后会引入一组肺癌患者与对照组尿样的开放数据并介绍数据特征；之后会应用最简单的t检验来寻找差异表达代谢物并讲解组学中的多重比较问题；最后结合患者的信息构建预测肺癌的统计学习模型，并讲解从单一模型到模型组合的数据分析过程。最终目的是使大家意识到实际数据分析的复杂性，并掌握一些处理实际问题的方法。

# 代谢组学简介

## 组学与代谢组学

组学就是研究生物体内各种组的生物学分支，这里组可以是基因组、转录组、蛋白质组或代谢组。组学的发展基本沿着中心法则来构建的，遗传信息可以从DNA构成的基因转录到RNA，RNA再翻译成蛋白质，因此了解基因组、转录组与蛋白质组可以搞清楚遗传信息的保存、调控与表达。同时，除了遗传信息，外界刺激也会影响生命过程的调控，遗传信息与外界刺激均会直接作用于代谢过程，因此代谢组作为了解复杂生命过程的一部分也受到更多关注。

![代谢组学与其他组学的关系](https://upload.wikimedia.org/wikipedia/commons/b/b8/The_central_dogma_of_biology_showing_the_flow_of_information_from_DNA_to_the_phenotype._Associated_with_each_stage_is_the_corresponding_systems_biology_tool_from_genomics_to_genomics_to_metabolomics.png)
图1. 代谢组学与其他组学的关系

不同于基因组、转录组与蛋白质组关注生物大分子，而代谢组学主要关注小分子代谢物（分子量小于1500Da）在生物体内变化过程，人类代谢组数据库（HMDB）收录了大概11万种已知小分子。同时，代谢组学除了研究已知代谢物还会覆盖未知代谢物，因此会分为针对已知代谢产物的目的性分析与针对未知代谢产物的非目的分析，可以类比基因组学中测定已知基因的芯片技术与对基因组进行完全测序，前者侧重于对已知的代谢产物或代谢通路在某个生命过程中的变化进行定量考察而后者会覆盖到未知的生物信息或者辅助研究人员产生假设并进行后续检验。目的性代谢组学是代谢过程传统研究方法的延伸而侧重假设检验而非目的代谢组学则更侧重探索性分析与系统分析。

## 代谢组学在医学中的应用

同其他组学类似，代谢组学在医学中的应用主要关注在疾病标记物与致病机理研究。疾病标记物可理解为某种疾病在潜伏期、发病期、恢复期等不同阶段的指示性物质，目前临床上已经大量应用疾病标记物进行诊断辅助，代谢组学更多致力于对前期症状不明显的疾病的筛查，由于代谢组学同时关注多种代谢物的变动模式而非单一物质，因此可借助机器学习模型更好提高标记准确度跟灵敏度。在致病机理上，代谢组学致力于发现疾病特有的代谢模式并进行机理解释，例如发现疾病特异性的代谢产物并推测上游生化反应的机理。

代谢组学在疾病机理研究中目前应用较多，主要用来探索疾病或某种健康状态的形成的分子机制或与外界刺激及遗传信息的关系，经常要结合其他组学技术系统考察。目前代谢组学的临床应用还处于早期，研究更侧重疾病标记物筛查，用研究中筛选到的小分子标记物来指示疾病的不同阶段。除了病理研究与临床诊断，代谢组学在预防医学与精准医疗中也受到关注，类似现在流行的个人基因检测，个人层面代谢过程经常与群体差异很大，如果能有效追踪个人样本例如血液中代谢组的变化，那么可针对个人提供个性化的治疗或保健方案。

## 代谢组学分析方法

代谢组学是现代分析技术特别是高分辨质谱色谱联用技术与核磁共振技术进步的产物，本教程只关注高分辨质谱色谱联用技术。在该分析技术中，生物样品的小分子代谢物经过有机溶剂萃取后可以送入仪器进行分离与定性定量分析。小分子代谢物的混合物经过色谱会分离开然后先后进入质谱，这会形成色谱峰信号，离子源会使代谢物电离带电，然后带电离子在不同质量分析器里的电场或磁场作用下产生信号或者说峰响应数据。也就是说，代谢物产生的峰响应数据会至少包括色谱分离产生的不同色谱峰保留时间（retention time，RT）、带电后产生的荷质比（m/z）及表示带电离子多少的响应值等，这些数据可以分离开不同的代谢物。

![原始数据](https://bookdown.org/yufree/Metabolomics/images/singledata.png)
图2. 高分辨质谱色谱联用技术收集到的原始信号，也就是不同保留时间带电离子在不同荷质比上的响应强度矩阵。

当我们关注到具体的某一个代谢产物的荷质比时，可以提取这个荷质比的色谱数据并进行峰拟合来获取峰面积积分作为峰的响应强度。

![峰原始数据](https://bookdown.org/yufree/Metabolomics/Metabolomics_files/figure-html/demoeic-1.png)
图3. 色谱峰的原始数据，通过平滑拟合后进行积分可得到这个色谱峰的响应强度

然而，因为同样的代谢物可能产生多个峰，所以色谱-质谱联用技术下的代谢组学数据存在冗余峰而不同的代谢物可能产生相同的峰，因此代谢组学也会涉及多级质谱数据，也就是代谢物电离后的离子可以进一步碎裂为带有结构信息的离子碎片，通过比对离子碎片峰与数据库可以对代谢物进行鉴定。本教程将不涉及多级质谱数据而仅关注一次电离后的峰数据，不过这类数据因为缺少代谢物结构鉴定步骤而属于非目的分析。

## 代谢组学数据结构

代谢组学的数据同其他组学数据类似，至少有两个部分，一部分是样本分组信息，来自于实验设计或调查问卷；另一部分则是每个样本中代谢组的信息，对于来自色谱-高分辨质谱联用仪器的数据，至少要包括三个部分：不同样本里峰响应数据、峰的荷质比与峰保留时间及样本的分组信息。从数据分析的角度，每个唯一的荷质比与保留时间峰的响应数据可看作每个生物样本的一维描述。一般而言，每个样本可以收集到成千上万的峰，也就是每个样本会有成千上万的来自代谢物的维度描述，但不要简单把代谢组学数据对应到其他组学，认为每一个峰都代表一种物质或一个独立维度，很多峰其实来自于同一物质，而数据分析的目标大都关心代谢物与样品分组信息的关系。通常我们会得到一个样本峰列表矩阵：

![样本峰列表](https://bookdown.org/yufree/Metabolomics/images/multidata.png)
图4. 样本峰列表示意图，这是符合通用机器学习的格式而生物信息学中通常会对这个列表进行转置。

下面我们通过一个案例来认识代谢组学的数据。

# 案例数据

## 案例背景介绍

肺癌曾经属于罕见疾病，但20世纪烟草行业的发展一举推高了肺癌的发病率，后续的研究发现不仅仅吸烟，氡气等因素也被认为是肺癌的来源，而也有证据支持少部分肺癌是由遗传因素决定的。本案例收集了 469 名肺癌患者与 536 名对照的尿液，利用色谱-高分辨质谱联用仪器收集了其中小分子信息，旨在寻找能指示肺癌的小分子标记物，本数据分为两份，来自于正负电离模式下同一样本的数据，这样做的主要原因是不同代谢物的物理化学性质不同，有的代谢物只在正离子或负离子模式下电离产生信号而有的代谢物则可以同时在两种电离模式下电离。这里我们用正离子模式来演示而负离子模式数据作为作业请同学们自由发挥进行分析。这组数据是网络公开数据，在metabolight上标号为mtbls28，可从[此处](https://www.ebi.ac.uk/metabolights/MTBLS28/descriptors)了解实验细节。

下面我们读入数据，这里需要注意的是为了演示这里用了处理后的csv文件，只包含了样本分组信息、峰响应数据、峰的荷质比与峰保留时间。此处峰的荷质比与峰保留时间对非目的分析很重要，例如剔除冗余峰或对峰的类型进行标注，这对理解真实数据很重要，这里我们只展示问题但在统计分析上并不会针对性处理，在面对真实数据时请一定注意咨询代谢组学的专家了解解决方案。

首先，我们用传统方法读入数据：

In [ ]:
mtbls28 <- read.csv('MTBLS28posmzrt.csv')

In [ ]:
# 观察数据
head(mtbls28)

这里我们看的csv里从第二行开始表示不同质谱峰，从第四列开始表示不同样本，csv的第二行表示的是样品分组信息，csv的第一列为峰标示名，第二列与第三列分别表示荷质比（mz）与保留时间（rt）。显然这样的格式并不标准，这里我们用`enviGCMS`包的`getmzrtcsv`函数来直接读入这个csv文件。

In [ ]:
mtbls28 <- enviGCMS::getmzrtcsv('MTBLS28posmzrt.csv')

In [ ]:
# 观察数据结构
str(mtbls28)

这里我们能看到这个csv文件被读入为`list`对象或`mzrt`对象，里面有四个元素：一个是含有1005样品中1806峰数据的名为 data 数据框，一个是含有样品名及分组信息的名为 group 的数据框，一个是含有1806个峰荷质比的名为 mz 的数字向量，一个是含有1806个峰保留时间名为 rt 的数字向量。此处 mz 跟 rt 的顺序与 data 的行顺序是一一对应的，group 的样品名与 data 的列顺序也是一一对应的，下面我们进行一些数据可视化。

## 数据可视化

我们首先检查分组信息，这里用直接用 `table` 函数：

In [ ]:
# 分组信息存在列表中的 group 元素中，group 是一个数据框，sample_group 这一列表示样品具体的分组信息
table(mtbls28$group$sample_group)

这里我们可以看到样品分组信息包括是否吸烟、种族、性别与疾病与否，这里我们只关心疾病与否。

然后，我们要可视化峰数据，也就是将荷质比与保留时间展示出来：

In [ ]:
# 这里列表对象中的mz表示荷质比，rt表示保留时间
plot(mtbls28$mz~mtbls28$rt,pch=19,xlab='retention time(s)',ylab='m/z')

此处我们会看到保留时间在0-400s之内，也就是说单个样品的分析时间不超过7分组，荷质比分布在1000以内，这里保留时间接近荷质比不同的峰可能是来自同一物质，也可能是色谱分离不了的多个物质，如果来自同一物质那么可以看做冗余峰。这里我们把所有峰看成不同的物质，但真实数据分析中是需要额外标注（annotation）处理的。

上述可视化只展示了峰之间的关系而无法体现样本间关系，然而1005个样本要展示1806维数据是一定要降维的，这里我们用主成分分析来做：

In [ ]:
# 这里分组信息用lv来输入
enviGCMS::plotpca(data = mtbls28$data, lv=mtbls28$group$sample_group)

这里我们可以看到前两个主成分只分别解释了不到10%的样本变异，此时降维意义有限，前两维的可视化可能无法展示出样品间差异。不过，如果主成分分析的前两个主成分能看出明显的样本分布差异且跟实验设计可以对应，那么说明捕捉的数据可以很好展示出实验设计的预设。在组学数据分析中，我们收集数据是不做预设的，其实收集的很多代谢物可能在人群里是没有差异的，这样的代谢物会让整体样本变异被随机变异主导，很难看到解释方差比较高的主成分。因此降维手段大都用来检查是否存在批次效应或样本异常值，经常是看不出明显的样本分布模式的，如果有的话多半不是生物来源的差异，有可能是数据采集质量不高。

## 差异分析：t检验分析单个代谢物组间差异

整体分析代谢物状况是一定需要降维的，但另一个思路就是逐个计算每个代谢物在不同分组间的差异，然后只去关心那些存在差异的代谢物。这其实就相当于对每一个代谢物单独进行差异分析，这里我们用 t 检验来演示第一个代谢峰在肺癌组与对照组的差异：

In [ ]:
# 利用正则表达式将控制组对照组进行区分并设定为分组信息
lv <- ifelse(grepl('Control',mtbls28$group$sample_group),'control','case')
# 对第一个峰进行t检验观察控制组对照组区别
t.test(as.numeric(mtbls28$data[1,])~lv)

这里我们可以看到这个峰在肺癌组与对照组是有显著性差异的，p值小于0.05.这样我们可以对1806个峰分别进行t检验，然后只关注那些有差异的峰。

## 多重比较问题：用q值替代p值控制假阳性代谢差异峰

对单一峰进行假设检验1806次是不同于只进行一次的，p值等于0.05就说明如果我们对两组没差异的随机数据进行100次t检验，随机作用下会有5次出现差异，也就是5%的可能性，如果你的假设检验只进行一次，那么还是可以接受的。然而，在我们这组数据里，我们进行了1806次假设检验，如果我们p值的阈值还是0.05，那么即使两组是没差异的，随机作用下也会出现90次差异，假设我们发现了100个峰在p值0.05阈值下有差异，那么其中大概一多半甚至全部都是假阳性。这个问题在组学研究中普遍存在，因此进行差异分析时通常我们会进行错误发现率（FDR）的整体控制，当我们同时进行的比较或假设检验很多的时候，判断是否出现差异的标准会变得更为严格来避免假阳性。一种相对通用的方法就是使用q值而不是p值来进行差异显著性判断，q值的算法有很多种但核心原则就是差异越大标准越严格，例如当q值的阈值设定为0.05时，我们得到的差异峰整体的错误发现率会控制在0.05，此时差异性峰的数目会减少但其中假阳性的峰也会减少。这里我们用演示数据来先计算p值，再根据p值计算q值。

In [ ]:
# 进行1806次t检验并将结果存为一个列表
re <- apply(mtbls28$data,1,function(x) t.test(x~lv))

In [ ]:
# 从列表中提取t检验的p值
pvalue <- sapply(re,function(x) x$p.value)
# 绘制p值直方图
hist(pvalue,breaks = 20)

这里我们可以看到p值的分布，大概可以看成一个均匀分布加一个真差异分布的联合分布，其中p值小于0.05的峰大概有800个，而因为其余p值大概是个均匀分布，因此这800个差异峰里面很可能有100个左右的假阳性。这里我们用BH矫正的q值计算方法来进行矫正：

In [ ]:
# 计算q值，这里用BH矫正方法
qvalue <- p.adjust(pvalue,method = 'BH')
# 计算q值小于0.05的峰数
sum(qvalue<0.05)

这里我们可以看到q值0.05阈值下有差异的峰为604个，这些更可能是真的差异峰。下一步可能就是对这些峰进行进一步鉴定，然后就可以拿来作为疾病标志物了。

# 预测模型：整合所有相关代谢峰的信息预测疾病

利用单个峰的差异分析来预测疾病与否是最简单的预测模型，如果想提高预测准确率，我们可以合并多个峰的信息进行预测。这里我们简单使用随机森林来构建一个简单的预测模型病对模型进行简单评价，之后我们会介绍下利用模型组合来提高预测准确率的方法。

## 随机森林模型：利用层级关系模拟代谢过程进行疾病预测

当我们构建疾病预测模型时，每一种代谢物都应该参与模型构建，因为代谢物之间天然存在一些基于代谢通路的关系，所以基于决策树的模型更有可能捕捉到这种关系。决策树模型会生成决策树，在不同节点参与决策的变量可能是不同的，决策树构建可通过引入重采样例如 bagging 提高模型稳健性，随机森林模型就是通过引入Bootstrap重采样过程外加强制使用较少的变量来实现的，可看作决策树模型的高阶版，当然也可以引入 Boosting 过程让早先生成的树基于已有的树成长，这里作为演示并不会涉及调参过程，也仅展示随机森林而不去讨论其他决策树模型。

作为机器学习或统计学习模型，构建过程最基本的要考虑训练集与验证集，建模过程要通过交叉检验来完成，这里我们使用`caret`包提供的机器学习框架，需要注意的是当前随机森林模型也许在医学还算比较新的模型，但在机器学习领域深度学习等基于人工神经网络的模型已经基本成为主流，当然因为不好用原理解释，有点用魔法攻克魔法的意味。

In [ ]:
install.package(h2o)

In [ ]:
# 这个包的教程可以参考这里：https://topepo.github.io/caret/
library(caret)
# 切割数据，这里我们一半用来建模，一半用来验证，用p来控制
trainIndex <- createDataPartition(lv, p = .5, 
                                  list = FALSE, 
                                  times = 1)
train <- mtbls28$data[, trainIndex]
train <- cbind.data.frame(Y=lv[trainIndex],t(train))
test  <- mtbls28$data[,-trainIndex]
test  <- cbind.data.frame(Y=lv[-trainIndex],t(test))

In [ ]:
# 训练集 10-fold 交叉检验
fit_control <- trainControl(method = "cv",
                           number = 10)
# 模型训练，这里rf就是用随机森林
mytrain <- train(Y~.,data=train,
                 method="rf",
                 trControl=fit_control,
                 standardize=T)
# 预览模型训练效果
plot(mytrain)

In [ ]:
# 观测模型效果
pROC::roc(test$Y, as.numeric(predict(mytrain, test[,-1], type = "raw")))

通过训练，我们可以得到一个验证集AUC在0.7的随机森林模型，而训练集上准确度在0.7以上，这说明我们构建了一个相对稳健的模型，此时我们可以把模型认为的重要变量进行输出，后续可以研究下这些代谢物的调控作用。这里的选择就跟前面差异分析里不一样了，差异分析中每个代谢物的计算都是单独的而在我们训练的模型之中代谢物的重要性是经过了无数次的重采用与交叉检验从整体层面得到的，后者更能反映代谢物的系统作用，但要进行机理解释与验证可能会比较困难。

In [ ]:
# 计算代谢峰对模型的重要性
rfImp <- varImp(mytrain, scale = FALSE)
rfImp

## 模型组合方法：利用多模型组合提高预测准确度

单一模型达到70%的准确度只能说过得去，但由于机器学习可以引入多种模型来捕捉数据中不同视角下的信息，所以通常我们还要用模型组合的方法来构建模型的模型来从工程手段上提高预测准确度。对于这组数据，我们使用的是随机森林，但也可以用线性判别分析来进行预测，这样在构建好两个模型后可以进一步训练各自模型预测的权重，这样就可能取长补短更好捕捉数据中的信息进行预测。

这里我们就用比较简单的 `caretEnsemble` 包来作为模型组合的框架，分别训练两个模型然后看看其组合后的预测效果。

In [ ]:
# 这里开下并行运算提速
library(doParallel)
# 设定CPU核数
registerDoParallel(2)
getDoParWorkers()
set.seed(123)
library(caretEnsemble)
# 5-fold 交叉检验 
my_control <- trainControl(method = "cv",
                           number = 5,
                           savePredictions = "final",
                           classProbs=TRUE,
                           allowParallel = TRUE)
# 提供需要训练的模型并进行训练，这里lda是线性判别分析，rf是随机森林
modellist <- caretList(Y~.,
                        data=train,
                        trControl = my_control,
                        methodList = c("lda",'rf'),
                        tuneList = NULL,
                        continue_on_fail = FALSE)
modellist
# 校对重采样结果
results <- resamples(modellist)
summary(results)

In [ ]:
# 利用训练好的模型在验证集上进行预测并对结果校对
model_preds <- lapply(modellist, predict, newdata=test, type="prob")
model_preds <- lapply(model_preds, function(x) x[,"case"])
model_preds <- data.frame(model_preds)
# 利用kappa系数作为评价标准对之前训练好的两个模型进行组合
greedy_ensemble <- caretEnsemble(
  modellist, 
  metric="kappa",
  trControl=trainControl(
    number=2,
    summaryFunction=twoClassSummary,
    classProbs=TRUE
    ))
# 用组合后的模型在验证集上进行预测
model_preds$ensemble <- predict(greedy_ensemble, newdata=test,type='prob')
# 比对单一模型与组合模型的效果
caTools::colAUC(model_preds, test$Y )

这里我们可以看到在验证集上模型组合后预测效果确有提升，但这里如果提升不明显很有可能是数据里信息已经被表现最好的模型捕捉到了，这种情况下可以尝试其他种类模型或提高样本量。

# 作业

前面用的只是尿样在正离子模式下的数据分析结果，请自行探索负离子模式下的代谢组学数据分析，如果学有余力，可以把正负离子模式的数据合并起来建模用来进行疾病预测。